# Phase 2 — Fixed Validation Split and F₀.₅ Scorer

This notebook creates a reproducible validation holdout at the **Source 1 entity** level before pair or candidate generation. It stratifies by the number of known matches (including singletons), selects exactly 20% within each stratum, and saves the validation Source 1 IDs to `dataset/splits/validation_source1_ids.txt`. The rest are the training partition. The raw data and labels are not modified.

The scorer below implements the README’s macro F₀.₅ definition per Source 1 record. An entity with no true matches and no predicted matches scores 1; if exactly one list is empty, it scores 0. It also reports the always-empty prediction baseline. Later experiments should generate candidates separately for the training and validation Source 1 partitions and keep this split fixed.


In [1]:
import csv, hashlib, json
from collections import Counter, defaultdict
from pathlib import Path

# Locate the project root from the current notebook directory.
search_roots = [Path.cwd(), *Path.cwd().parents]
project_root = next((p for p in search_roots if (p / "dataset" / "train" / "train_ground_truth.tsv").is_file()), None)
if project_root is None:
    raise FileNotFoundError("Could not locate dataset/train/train_ground_truth.tsv")
ground_truth_path = project_root / "dataset" / "train" / "train_ground_truth.tsv"
split_dir = project_root / "dataset" / "splits"
split_dir.mkdir(parents=True, exist_ok=True)
validation_ids_path = split_dir / "validation_source1_ids.txt"
manifest_path = split_dir / "phase2_validation_manifest.json"
seed_namespace = 'ML-CHALLENGE-2026-PHASE2-SEED-1'
validation_fraction = 0.2

# Group Source 1 IDs by true match-list size, then take an exact, deterministic
# 20% sample per stratum using SHA-256 rank. This keeps each match-count stratum
# represented in both partitions and does not depend on Python's random module.
groups = defaultdict(list)
with ground_truth_path.open(encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f, delimiter="\t")
    if reader.fieldnames != ["source1_entity_id", "matched_entity_ids"]:
        raise ValueError(f"Unexpected ground-truth columns: {reader.fieldnames}")
    for row in reader:
        source1_id = row["source1_entity_id"].strip()
        match_ids = [value.strip() for value in (row["matched_entity_ids"] or "").split(",") if value.strip()]
        groups[len(match_ids)].append(source1_id)

validation_ids = set()
train_distribution, validation_distribution = Counter(), Counter()
for match_count, source_ids in groups.items():
    if len(source_ids) < 2:
        raise ValueError(f"Stratum {match_count} is too small for a train/validation split")
    source_ids.sort(key=lambda source_id: hashlib.sha256(f"{seed_namespace}|{source_id}".encode("utf-8")).digest())
    validation_count = max(1, int(len(source_ids) * validation_fraction + 0.5))
    validation_count = min(validation_count, len(source_ids) - 1)
    validation_ids.update(source_ids[:validation_count])
    validation_distribution[match_count] = validation_count
    train_distribution[match_count] = len(source_ids) - validation_count

with validation_ids_path.open("w", encoding="utf-8", newline="\n") as f:
    f.write("source1_entity_id\n")
    for source_id in sorted(validation_ids):
        f.write(source_id + "\n")
manifest = {
    "purpose": "fixed Source 1 holdout for validation; training is the complement",
    "seed_namespace": seed_namespace,
    "split_method": "SHA-256 rank within true match-count strata",
    "requested_validation_fraction": validation_fraction,
    "ground_truth_rows": sum(map(len, groups.values())),
    "train_rows": sum(train_distribution.values()),
    "validation_rows": len(validation_ids),
    "train_match_count_distribution": dict(sorted(train_distribution.items())),
    "validation_match_count_distribution": dict(sorted(validation_distribution.items())),
    "validation_ids_file": str(validation_ids_path.relative_to(project_root)),
}
manifest_path.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
print(json.dumps(manifest, indent=2))


{
  "purpose": "fixed Source 1 holdout for validation; training is the complement",
  "seed_namespace": "ML-CHALLENGE-2026-PHASE2-SEED-1",
  "split_method": "SHA-256 rank within true match-count strata",
  "requested_validation_fraction": 0.2,
  "ground_truth_rows": 2206821,
  "train_rows": 1765458,
  "validation_rows": 441363,
  "train_match_count_distribution": {
    "0": 98598,
    "1": 95326,
    "2": 300170,
    "3": 424673,
    "4": 387292,
    "5": 257566,
    "6": 131894,
    "7": 51174,
    "8": 14944,
    "9": 3364,
    "10": 427,
    "11": 30
  },
  "validation_match_count_distribution": {
    "0": 24649,
    "1": 23831,
    "2": 75042,
    "3": 106168,
    "4": 96823,
    "5": 64391,
    "6": 32974,
    "7": 12794,
    "8": 3736,
    "9": 841,
    "10": 107,
    "11": 7
  },
  "validation_ids_file": "dataset\\splits\\validation_source1_ids.txt"
}


In [2]:
from collections import Counter
import csv


def entity_f05(true_ids, predicted_ids):
    """README F_0.5 score for one Source 1 entity, including the singleton rule."""
    true_set, predicted_set = set(true_ids), set(predicted_ids)
    if not true_set and not predicted_set:
        return 1.0
    tp = len(true_set & predicted_set)
    fp = len(predicted_set - true_set)
    fn = len(true_set - predicted_set)
    denominator = 1.25 * tp + fp + 0.25 * fn
    return (1.25 * tp / denominator) if denominator else 0.0


def macro_f05(truth_by_source1, predictions_by_source1):
    """Macro-average entity F_0.5; require exactly one prediction per truth row."""
    if truth_by_source1.keys() != predictions_by_source1.keys():
        missing = truth_by_source1.keys() - predictions_by_source1.keys()
        extra = predictions_by_source1.keys() - truth_by_source1.keys()
        raise ValueError(f"Prediction key mismatch: {len(missing)} missing, {len(extra)} extra")
    if not truth_by_source1:
        raise ValueError("Cannot score an empty validation set")
    scores = [entity_f05(truth_by_source1[source_id], predictions_by_source1[source_id])
              for source_id in truth_by_source1]
    return sum(scores) / len(scores)


project_root = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "dataset" / "train" / "train_ground_truth.tsv").is_file())
validation_ids_path = project_root / "dataset" / "splits" / "validation_source1_ids.txt"
validation_ids = set(validation_ids_path.read_text(encoding="utf-8").splitlines()[1:])
truth = {}
with (project_root / "dataset" / "train" / "train_ground_truth.tsv").open(encoding="utf-8", newline="") as f:
    for row in csv.DictReader(f, delimiter="\t"):
        source_id = row["source1_entity_id"]
        if source_id in validation_ids:
            truth[source_id] = [value for value in row["matched_entity_ids"].split(",") if value]
if truth.keys() != validation_ids:
    raise ValueError(f"Validation labels missing: {len(validation_ids - truth.keys())}")

# Baseline: predict no matches. Singleton rows score 1; all other rows score 0.
empty_predictions = {source_id: () for source_id in truth}
no_match_baseline = macro_f05(truth, empty_predictions)
singletons = sum(not ids for ids in truth.values())
print(f"Validation Source 1 rows: {len(truth):,}")
print(f"Validation singletons: {singletons:,} ({singletons / len(truth):.4%})")
print(f"No-match baseline macro F_0.5: {no_match_baseline:.6f}")


Validation Source 1 rows: 441,363
Validation singletons: 24,649 (5.5847%)
No-match baseline macro F_0.5: 0.055847
